# 01 — Auditoría de datos (plan reproducible, sin resultados inventados)

Estado: **PENDIENTE DE EJECUCIÓN**. Falta configurar `.env` con `RAWG_API_KEY`, `IGDB_CLIENT_ID`, `IGDB_CLIENT_SECRET` (ver `.env.example`).

Este notebook no contiene estadísticas. Cuando haya credenciales, ejecútalo de arriba abajo y generará la muestra PC / consola / móvil + auditoría C1–C7.

Documentos: `docs/01_validacion_fuentes.md` (endpoints y condiciones 2026-10-13) y `docs/02_decision_fuentes.md` (criterios y decisión IGDB-principal).

In [ ]:
import os
import time
import json
from datetime import datetime, timezone
from pathlib import Path

import requests
import pandas as pd
from dotenv import load_dotenv

# Cargamos .env local sin exponer secretos en el código.
load_dotenv()
RAWG_KEY = os.getenv("RAWG_API_KEY", "")
IGDB_ID = os.getenv("IGDB_CLIENT_ID", "")
IGDB_SECRET = os.getenv("IGDB_CLIENT_SECRET", "")

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("RAWG configurada:", bool(RAWG_KEY))
print("IGDB configurada:", bool(IGDB_ID and IGDB_SECRET))
print("Fecha UTC:", datetime.now(timezone.utc).isoformat())

In [ ]:
def _get(url, params=None, headers=None, timeout=20):
    """GET con manejo explícito de 401/404/429/5xx para no confundir un bloqueo con un nulo."""
    try:
        r = requests.get(url, params=params, headers=headers, timeout=timeout)
    except requests.RequestException as e:
        return {"ok": False, "error": f"red: {e}"}
    if r.status_code == 429:
        return {"ok": False, "error": "429 rate limit: esperar y reintentar con backoff"}
    if r.status_code in (401, 403):
        return {"ok": False, "error": f"{r.status_code} auth: revisar key/token"}
    if r.status_code >= 500:
        return {"ok": False, "error": f"{r.status_code} servidor: reintentar con jitter"}
    if r.status_code != 200:
        return {"ok": False, "error": f"{r.status_code}: {r.text[:300]}"}
    try:
        return {"ok": True, "data": r.json()}
    except ValueError:
        return {"ok": False, "error": "respuesta no JSON"}


def fetch_rawg_platforms(page_size=40):
    """Descubre IDs de plataforma en lugar de hardcodearlos: evita asumir que 18/1/7 siguen vigentes."""
    if not RAWG_KEY:
        return {"ok": False, "error": "falta RAWG_API_KEY en .env"}
    return _get("https://api.rawg.io/api/platforms", params={"key": RAWG_KEY, "page_size": page_size})


def fetch_rawg_games(page_size=20, platforms=None, search=None):
    """Muestra pequeña y trazable. Throttle manual porque el plan Free es 20k/mes."""
    if not RAWG_KEY:
        return {"ok": False, "error": "falta RAWG_API_KEY en .env"}
    params = {"key": RAWG_KEY, "page_size": page_size}
    if platforms:
        params["platforms"] = platforms
    if search:
        params["search"] = search
        params["search_precise"] = True
    time.sleep(1.2)  # evita acercarse al rate limit en pruebas secuenciales
    return _get("https://api.rawg.io/api/games", params=params)


def get_igdb_token():
    """Token Twitch OAuth2 client_credentials (~60 días). Se renueva, no se versiona."""
    if not (IGDB_ID and IGDB_SECRET):
        return {"ok": False, "error": "faltan IGDB_CLIENT_ID/SECRET en .env"}
    try:
        r = requests.post(
            "https://id.twitch.tv/oauth2/token",
            params={"client_id": IGDB_ID, "client_secret": IGDB_SECRET, "grant_type": "client_credentials"},
            timeout=20,
        )
    except requests.RequestException as e:
        return {"ok": False, "error": f"red token: {e}"}
    if r.status_code != 200:
        return {"ok": False, "error": f"token {r.status_code}: {r.text[:300]}"}
    return {"ok": True, "data": r.json()}


def igdb_post(endpoint, body, token, retries=3):
    """POST Apicalypse con backoff ante 429 y reintento ante 5xx. IGDB no admite GET para consultas."""
    url = f"https://api.igdb.com/v4/{endpoint}"
    headers = {"Client-ID": IGDB_ID, "Authorization": f"Bearer {token}"}
    for i in range(retries):
        try:
            r = requests.post(url, headers=headers, data=body, timeout=30)
        except requests.RequestException as e:
            time.sleep(2 ** i)
            err = f"red: {e}"
            continue
        if r.status_code == 429:
            time.sleep(2 ** i + 1)  # respeta 4 req/s, hasta 8 abiertas
            err = "429 rate limit"
            continue
        if r.status_code == 401:
            return {"ok": False, "error": "401 token caducado: regenerar token"}
        if r.status_code >= 500:
            time.sleep(2 ** i)
            err = f"{r.status_code} servidor"
            continue
        if r.status_code != 200:
            return {"ok": False, "error": f"{r.status_code}: {r.text[:400]}"}
        try:
            return {"ok": True, "data": r.json()}
        except ValueError:
            return {"ok": False, "error": "respuesta no JSON"}
        time.sleep(0.3)  # mantiene <4 req/s en secuencial
    return {"ok": False, "error": err}


In [ ]:
def audit_dataframe(df, name):
    """Auditoría C1–C7 sin eliminar nada: solo mide y explica el impacto posterior."""
    print(f"\n===== {name} =====")
    print("Registros:", len(df))
    if df.empty:
        print("Vacío: no se puede concluir cobertura. Revisar auth, filtros o rate limit.")
        return
    # % nulos por campo: decide qué KPIs serán fiables en la app
    print("\n% nulos por campo:")
    print((df.isna().mean() * 100).round(2).sort_values(ascending=False))
    # Duplicados por clave natural: romperían el comparador y los snapshots
    key = [c for c in ["source", "source_id", "platform_id"] if c in df.columns]
    if key:
        print(f"\nDuplicados por {key}:", int(df.duplicated(subset=key).sum()))
    # IDs ausentes: impiden trazabilidad y joins con snapshots
    for c in ["source_id", "name"]:
        if c in df.columns:
            print(f"Ausentes {c}:", int(df[c].isna().sum()))
    # Cobertura y disponibilidad de métricas: alimenta C1, C3, C4
    for c in ["platform_name", "genre", "release_date", "critic_score", "user_score", "price"]:
        if c in df.columns:
            print(f"Disponible {c}: {(1 - df[c].isna().mean()) * 100:.1f}%")


In [ ]:
observed_at = datetime.now(timezone.utc).isoformat()

if not RAWG_KEY and not (IGDB_ID and IGDB_SECRET):
    print("PENDIENTE: configura .env desde .env.example y vuelve a ejecutar.")
    print("Plan: 1) GET /platforms (RAWG) y POST /platforms (IGDB) para mapear PC/consola/móvil.")
    print("      2) 3 muestras de >=30 juegos (PC, consola, móvil) con page_size>=20 + paginación.")
    print("      3) Guardar raw IGDB en data/raw/igdb_*.json con observed_at+source; raw RAWG solo local gitignorado.")
    print("      4) Normalizar a columnas: source,source_id,name,platform_id,platform_name,genre,release_date,developer,publisher,critic_score,user_score,ratings_count,price,currency,observed_at.")
    print("      5) audit_dataframe() por fuente y decidir C1–C7. No inventar cifras.")
else:
    # --- Ejemplo mínimo RAWG (descomentar tras mapear platforms) ---
    # plat = fetch_rawg_platforms()
    # print(plat["ok"], str(plat)[:500])
    # sample = fetch_rawg_games(page_size=20)
    # if sample["ok"]:
    #     rows = [{"source": "rawg", "source_id": g.get("id"), "name": g.get("name"),
    #              "release_date": g.get("released"), "critic_score": g.get("metacritic"),
    #              "user_score": g.get("rating"), "ratings_count": g.get("ratings_count"),
    #              "observed_at": observed_at} for g in sample["data"].get("results", [])]
    #     audit_dataframe(pd.DataFrame(rows), "RAWG muestra")
    # else: print("RAWG error:", sample["error"])

    # --- Ejemplo mínimo IGDB (descomentar con token válido) ---
    # tok = get_igdb_token()
    # if tok["ok"]:
    #     token = tok["data"]["access_token"]
    #     res = igdb_post("games", "fields id,name,genres.name,platforms.name,first_release_date,aggregated_rating,aggregated_rating_count,rating,rating_count; limit 20;", token)
    #     print(res["ok"], str(res["data"][:1])[:800] if res["ok"] else res["error"])
    # else: print("IGDB error:", tok["error"])
    print("Listo para ejecutar cuando descomentes el bloque con credenciales válidas.")

## Qué falta para C1–C7

- Mapear IDs RAWG (`GET /platforms`) y familias IGDB (`POST /platform_families, /platforms`) a PC / consola / móvil.
- Medir % nulos, duplicados, cobertura y disponibilidad de `metacritic/aggregated_rating` vs `rating`.
- Guardar `observed_at` + `source` en cada fila; no presentar datos actuales como históricos.
- Decidir en `docs/02_decision_fuentes.md` si se mantiene IGDB-principal o se invierte, con cifras reales.